In [1]:


import pandas as pd, numpy as np, requests, time, torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

df = pd.read_csv("kcnq1_coords.csv", dtype={"chrom": str})
HALF = 1000   # 2001 bp window, about 334 tokens of 6-mers

def fetch(chrom, pos, tries=5):
    url = f"https://rest.ensembl.org/sequence/region/human/{chrom}:{pos-HALF}..{pos+HALF}:1"
    for a in range(tries):
        try:
            r = requests.get(url, headers={"Content-Type": "text/plain"}, timeout=60)
            r.raise_for_status()
            return r.text.strip().upper()
        except requests.RequestException:
            time.sleep(5 * 2**a)
    raise RuntimeError(f"Ensembl failed for {chrom}:{pos}")

name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"
tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(name, trust_remote_code=True).eval()
dev = "cuda" if torch.cuda.is_available() else "cpu"
model.to(dev)

@torch.no_grad()
def llr(ref_seq, alt_seq):
    ids_r = tok(ref_seq, return_tensors="pt")["input_ids"][0]
    ids_a = tok(alt_seq, return_tensors="pt")["input_ids"][0]
    diff = (ids_r != ids_a).nonzero().flatten()
    if len(diff) != 1:
        return None                      # an SNV should change exactly one 6-mer token
    i = diff.item()
    masked = ids_r.clone()
    masked[i] = tok.mask_token_id        # hide that token, ask the model to predict it
    logits = model(masked.unsqueeze(0).to(dev)).logits[0, i]
    lp = torch.log_softmax(logits, -1)
    return (lp[ids_a[i]] - lp[ids_r[i]]).item()   # log P(alt) - log P(ref)

scores, problems = [], []
for _, r in df.iterrows():
    ref_seq = fetch(r.chrom, int(r.pos))
    if len(ref_seq) != 2*HALF + 1 or ref_seq[HALF] != r.ref:
        problems.append(r["name"]); scores.append(None); continue
    alt_seq = ref_seq[:HALF] + r.alt + ref_seq[HALF+1:]
    scores.append(llr(ref_seq, alt_seq))
    time.sleep(0.3)

df["nt_llr"] = scores
print("Scored:", df.nt_llr.notna().sum(), "/", len(df), "| Problems:", problems)
df.to_csv("kcnq1_lm_scores.csv", index=False)

from google.colab import files
files.download("kcnq1_lm_scores.csv")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.99G [00:00<?, ?B/s]

Scored: 47 / 47 | Problems: []


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
!pip install -q "transformers==4.46.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import json, os
cache_file = "windows.json"
wins = json.load(open(cache_file)) if os.path.exists(cache_file) else {}
for _, r in df.iterrows():
    k = f"{r.chrom}:{r.pos}"
    if k not in wins:
        wins[k] = fetch(r.chrom, int(r.pos))
        json.dump(wins, open(cache_file, "w"))
        time.sleep(0.3)

models = {
    "nt_human500": "InstaDeepAI/nucleotide-transformer-500m-human-ref",
    "nt_v2_100m":  "InstaDeepAI/nucleotide-transformer-v2-100m-multi-species",
}
for col, name in models.items():
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(name, trust_remote_code=True).eval().to(dev)
    out = []
    for _, r in df.iterrows():
        ref_seq = wins[f"{r.chrom}:{r.pos}"]
        alt_seq = ref_seq[:HALF] + r.alt + ref_seq[HALF+1:]
        out.append(llr(ref_seq, alt_seq))   # llr() uses the global tok/model set just above
    df[col] = out
    print(col, "scored:", df[col].notna().sum(), "/", len(df))
    del model; torch.cuda.empty_cache()

df.to_csv("kcnq1_lm_scores.csv", index=False)
files.download("kcnq1_lm_scores.csv")

tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-100m-multi-species:
- modeling_esm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/392M [00:00<?, ?B/s]

nt_v2_100m scored: 47 / 47


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>